In [1]:
import pandas as pd

import torch

## Load Datasets

In [2]:
df = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

df.shape, df_test.shape

((891, 12), (418, 11))

In [3]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


## NULL Values

In [4]:
pd.DataFrame({'train': df.isna().sum(), 'test': df_test.isna().sum()})

,train,test
Age,177,86.0
Cabin,687,327.0
Embarked,2,0.0
Fare,0,1.0
Name,0,0.0
Parch,0,0.0
PassengerId,0,0.0
Pclass,0,0.0
Sex,0,0.0
SibSp,0,0.0


### Age

In [5]:
df[['Age']].describe()

,Age
count,714.000000
mean,29.699118
std,14.526497
min,0.420000
25%,20.125000
50%,28.000000
75%,38.000000
max,80.000000


### Fare

In [6]:
df[['Fare']].describe()

,Fare
count,891.000000
mean,32.204208
std,49.693429
min,0.000000
25%,7.910400
50%,14.454200
75%,31.000000
max,512.329200


### Cabin

In [7]:
df['Cabin'].value_counts()

Cabin
G6             4
C23 C25 C27    4
B96 B98        4
F33            3
E101           3
              ..
E17            1
A24            1
C50            1
B42            1
C148           1
Name: count, Length: 147, dtype: int64

### Embarked

In [8]:
df['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

## Preprocessing

`add_features` holds the row-wise steps that need no statistics. `Preprocessor.fit` learns every statistic (fill values, means/stds, one-hot columns) from the data it is fitted on, and `transform` applies them unchanged to any other split.

In [9]:
def add_features(df):
    df = df.copy()

    df['Deck'] = df['Cabin'].str.split().str[-1].str[0].fillna('unknown')
    df['CabinCount'] = df['Cabin'].str.split().str.len().fillna(0).astype(int)

    return df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived'], errors='ignore')

In [10]:
class Preprocessor:
    def fit(self, df):
        X = add_features(df)

        self.age_mean = X['Age'].mean()
        self.fare_median = X['Fare'].median()
        self.embarked_mode = X['Embarked'].mode()[0]
        X = self._impute(X)

        self.numeric_cols = X.select_dtypes(include='number').columns.tolist()
        self.categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

        self.means = X[self.numeric_cols].mean()
        self.stds = X[self.numeric_cols].std()

        self.columns = pd.get_dummies(X, columns=self.categorical_cols).columns
        
        return self

    def transform(self, df):
        X = self._impute(add_features(df))

        X[self.numeric_cols] = (X[self.numeric_cols] - self.means) / self.stds

        X = pd.get_dummies(X, columns=self.categorical_cols, dtype=int)
        
        return X.reindex(columns=self.columns, fill_value=0)

    def _impute(self, X):
        return X.fillna({'Age': self.age_mean, 'Fare': self.fare_median, 'Embarked': self.embarked_mode})

In [11]:
def to_tensor(data):
    return torch.tensor(data.to_numpy(), dtype=torch.float32)


def to_labels(df):
    return torch.tensor(df['Survived'].to_numpy(), dtype=torch.int64)

### Engineered Features

In [12]:
features = add_features(df)

features[['Deck']].describe()

,Deck
count,891
unique,9
top,unknown
freq,687


In [13]:
features['CabinCount'].value_counts()

CabinCount
0    687
1    180
2     16
3      6
4      2
Name: count, dtype: int64

## Train/Val Split

Statistics are fitted on the 80% train split only and applied to both splits.

In [14]:
df_val = df.groupby('Survived').sample(frac=0.2, random_state=42)
df_train = df.drop(index=df_val.index)

len(df_train), len(df_val)

(713, 178)

In [15]:
split_prep = Preprocessor().fit(df_train)

df_train_X = split_prep.transform(df_train)
df_val_X = split_prep.transform(df_val)

df_train_X.shape, df_val_X.shape

((713, 20), (178, 20))

In [16]:
df_train_X[split_prep.numeric_cols].describe()

,Pclass,Age,SibSp,Parch,Fare,CabinCount
count,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02
mean,1.918366e-16,-1.893452e-16,9.965536e-18,7.162729e-17,2.491384e-18,7.474152e-18
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.571832e+00,-2.216675e+00,-4.862172e-01,-4.886608e-01,-6.545193e-01,-4.775515e-01
25%,-3.680553e-01,-5.515305e-01,-4.862172e-01,-4.886608e-01,-4.961683e-01,-4.775515e-01
50%,8.357219e-01,-2.741326e-16,-4.862172e-01,-4.886608e-01,-3.657070e-01,-4.775515e-01
75%,8.357219e-01,4.515686e-01,4.260798e-01,-4.886608e-01,-2.735935e-02,-4.775515e-01
max,8.357219e-01,3.460866e+00,6.812158e+00,7.311679e+00,9.582433e+00,6.579324e+00


In [17]:
torch.save({
    'X_train': to_tensor(df_train_X),
    'Y_train': to_labels(df_train),
    'X_val': to_tensor(df_val_X),
    'Y_val': to_labels(df_val),
    'columns': df_train_X.columns.tolist(),
}, '../data/train_val.pt')

## Full Train and Test

Statistics are fitted on all of `train.csv` and applied to `test.csv`, so a model trained on `train_full.pt` matches `test.pt`.

In [18]:
full_prep = Preprocessor().fit(df)

df_full_X = full_prep.transform(df)
df_test_X = full_prep.transform(df_test)

df_full_X.shape, df_test_X.shape

((891, 20), (418, 20))

In [19]:
torch.save({
    'X': to_tensor(df_full_X),
    'Y': to_labels(df),
    'columns': df_full_X.columns.tolist(),
}, '../data/train_full.pt')

torch.save({
    'X': to_tensor(df_test_X),
    'PassengerId': torch.tensor(df_test['PassengerId'].to_numpy()),
    'columns': df_test_X.columns.tolist(),
}, '../data/test.pt')

## Check Saved Tensors

In [20]:
for name in ('train_val', 'train_full', 'test'):
    data = torch.load(f'../data/{name}.pt')
    shapes = {k: tuple(v.shape) if torch.is_tensor(v) else len(v) for k, v in data.items()}
    nans = sum(v.isnan().sum().item() for v in data.values() if torch.is_tensor(v) and v.is_floating_point())
    print(name, shapes, 'NaNs:', nans)

train_val {'X_train': (713, 20), 'Y_train': (713,), 'X_val': (178, 20), 'Y_val': (178,), 'columns': 20} NaNs: 0
train_full {'X': (891, 20), 'Y': (891,), 'columns': 20} NaNs: 0
test {'X': (418, 20), 'PassengerId': (418,), 'columns': 20} NaNs: 0
